# Normalize Morita et al. lipid names to SwissLipids format

**Goal:** Match lipid names from Morita et al. to SwissLipids entries (SL_ID + SMILES)
using a cascade of name-format conversions.
SwissLipids is used as a complementary database to LipidMaps, especially for
lipid species absent from LipidMaps (e.g. Cer with PUFA acyl chains).

**Upstream:** `norm_data_goslin.ipynb` → `goslin_Morita.csv` (GOSLIN-normalized names)  
**Output:** `data/processed/swisslipids_Morita.csv`

**`_` vs `/` (LIPID MAPS Shorthand Nomenclature):**
- `_` = fatty acid composition known, sn-position **unknown** (Morita raw data uses `_` exclusively)
- `/` = sn-position **confirmed** — not introduced in intermediate names for glycerophospholipids
- **Sphingolipid exception:** SwissLipids stores Cer/SM/HexCer only with `/` in d-notation (0 `_` entries).
  The `_`→`/` conversion in `to_cer_d_slash` is intentional — see code comment for biological justification.

**Key format differences:**

| Level | Morita (raw) | Intermediate | SwissLipids (DB key) | Sep rule |
|---|---|---|---|---|
| Molecular subspecies | `PC 16:0_18:1` | `PC 16:0_18:1` | `PC(16:0_18:1)` | `_` preserved |
| Ceramide | `Cer d18:1_16:0` | `Cer 18:1;2_16:0` | `Cer(d18:1/16:0)` | `_`→`/` (sphingolipid exception) |
| Lyso | `LPC 16:0` | `LPC 16:0` | `LPC(16:0_0:0)` | `0:0` appended |

**Steps:**
1. Download SwissLipids library via pypath (Abbreviation + Synonyms indexed)
2. Load Morita data, drop dummy rows (ExactMass filter)
3. Normalize Morita names to intermediate format; load goslin_Morita.csv
4. Cascade matching (5 strategies adapted for SwissLipids format)
5. Report match rate and compare with LipidMaps results
6. Save results
7. SwissLipids hierarchy level breakdown


In [39]:
import re
import pandas as pd
from collections import Counter

## Step 1 — Download SwissLipids library via pypath

Build `name → list of {sl_id, smiles, formula}` from SwissLipids.
Both `Abbreviation*` and `Synonyms*` fields are indexed as keys.

**Level filter:** We include all levels with SMILES available.
SwissLipids levels (largest to smallest):
- `Isomeric subspecies` — full stereo info
- `Structural subspecies` — sn-position known
- `Molecular subspecies` — chain composition known, sn unknown
- `Species` — only class + total C:DB known

In [40]:
import os, pickle

CACHE_PATH = '../data/processed/cache_swisslipids_db.pkl'

if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, 'rb') as f:
        lipid_db = pickle.load(f)
    print(f'Loaded from cache: {len(lipid_db):,} keys  ({CACHE_PATH})')
    print('Delete the cache file to force a fresh download.')
else:
    from pypath.inputs_v2.swisslipids import resource

    # name -> list of {'sl_id': ..., 'smiles': ..., 'formula': ..., 'level': ...}
    lipid_db = {}
    n_skipped_no_smiles = 0

    for raw in resource.lipids.raw():
        sl_id  = raw.get('Lipid ID', '').strip()
        smiles = raw.get('SMILES (pH7.3)', '').strip()
        if not sl_id or not smiles:
            n_skipped_no_smiles += 1
            continue

        formula = raw.get('Formula (pH7.3)', '').strip()
        level   = raw.get('Level', '').strip()
        entry   = {'sl_id': sl_id, 'smiles': smiles, 'formula': formula, 'level': level}

        # Collect all name variants for this entry
        names = set()
        abbr  = raw.get('Abbreviation*', '').strip()
        syns  = raw.get('Synonyms*', '')

        if abbr:
            for a in abbr.split('|'):
                names.add(a.strip())
        if syns:
            for s in syns.split(';'):
                names.add(s.strip())

        for name in names:
            if name:
                lipid_db.setdefault(name, []).append(entry)

    with open(CACHE_PATH, 'wb') as f:
        pickle.dump(lipid_db, f)

    print(f'Built and cached   : {len(lipid_db):,} keys')
    print(f'Cache saved to     : {CACHE_PATH}')
    print(f'Entries skipped (no SMILES): {n_skipped_no_smiles:,}')

print(f'Example keys: {list(lipid_db.keys())[200:205]}')

Loaded from cache: 1,252,538 keys  (../data/processed/cache_swisslipids_db.pkl)
Delete the cache file to force a fresh download.
Example keys: ['PA(O-)', 'Stearyl alcohol', 'Cer(d18:0/22:0)', 'Ceramide (d18:0/22:0)', 'Ceramide (d18:0/18:0)']


## Step 1.5 — Load Morita data and drop dummy rows

Rows where `ExactMass` is empty are not measured lipids but dummy/placeholder rows.

In [41]:
df_raw = pd.read_excel('../data/raw/lipid_Morita_et_al.xlsx', sheet_name='Lipid_all')

n_before = len(df_raw)
df_raw   = df_raw[df_raw['ExactMass'].notna()].reset_index(drop=True)
n_after  = len(df_raw)

print(f'Rows before filtering : {n_before}')
print(f'Rows dropped (dummy)  : {n_before - n_after}')
print(f'Rows after filtering  : {n_after}')

Rows before filtering : 1306
Rows dropped (dummy)  : 17
Rows after filtering  : 1289


## Step 2 — Normalize Morita names and load GOSLIN reference

Apply two normalization rules to convert Morita raw names to an intermediate format.

**`_` vs `/`:** Morita data uses `_` exclusively (sn-position unknown).
`_` is preserved throughout — it is never converted to `/` in the intermediate representation.

| Rule | Input | Output | Note |
|---|---|---|---|
| Parenthesis chains | `TG(12:0)(14:1)(18:2)` | `TG 12:0_14:1_18:2` | `_` preserved |
| Ceramide `d` prefix | `Cer d18:1_16:0` | `Cer 18:1;2_16:0` | `d`→`;2` is equivalent; `_` preserved |

Rule 3 (underscore → slash) has been **removed**: it fabricated sn-position information absent from the raw data.

Also loads `goslin_Morita.csv` to use GOSLIN-parsed class names as additional matching keys.

In [42]:
def normalize_lipid_name(name):
    """
    Convert Morita et al. proprietary lipid names to an intermediate format
    for SwissLipids cascade matching.

    '_' vs '/':
      '_' = fatty acid composition known, sn-position UNKNOWN  (Morita raw data)
      '/' = sn-position CONFIRMED — must NOT be introduced unless present in raw data

    Rules applied in order:
    1. Parenthesis chains -> underscore-separated (molecular species level)
       e.g. TG(12:0)(14:1)(18:2) -> TG 12:0_14:1_18:2
    2. Ceramide 'd' prefix -> ';2' hydroxyl notation  (equivalent, not fabrication)
       e.g. Cer d18:1_16:0 -> Cer 18:1;2_16:0
       '_' is preserved as chain separator (NOT converted to '/')
    """
    # Rule 1: parenthesis-enclosed chains — use '_', not '/'
    m = re.match(r'^([A-Za-z][A-Za-z0-9]*)((?:\([^)]+\))+)$', name)
    if m:
        cls    = m.group(1)
        chains = re.findall(r'\(([^)]+)\)', m.group(2))
        name   = cls + ' ' + '_'.join(chains)

    # Rule 2: Ceramide 'd' prefix — preserve '_' as chain separator
    name = re.sub(r'((?:Hex)?Cer) d(\d+:\d+)_(\S+)', r'\1 \2;2_\3', name)

    return name


lipid_names      = df_raw['CompoundName'].dropna().unique().tolist()
lipid_names_norm = [normalize_lipid_name(n) for n in lipid_names]

# Load GOSLIN-normalized reference for species-level fallback
try:
    df_goslin = pd.read_csv('../data/processed/goslin_Morita.csv')[
        ['original_name', 'class_name', 'formula']
    ].rename(columns={'formula': 'formula_goslin'})
    goslin_formula = dict(zip(df_goslin['original_name'], df_goslin['formula_goslin']))
    print('goslin_Morita.csv loaded for formula-level fallback.')
except FileNotFoundError:
    goslin_formula = {}
    print('goslin_Morita.csv not found — formula fallback disabled.')

print(f'Total unique Morita lipid names: {len(lipid_names)}')
print()
print('Normalization examples:')
for orig, norm in zip(lipid_names[:6], lipid_names_norm[:6]):
    print(f'  {orig:35s} -> {norm}')

# Verify: no '_'-containing original produces a '/'-containing normalized name
violations_norm = [
    (o, n) for o, n in zip(lipid_names, lipid_names_norm)
    if '_' in o and '/' in n
]
if violations_norm:
    print(f'\nWARNING: {len(violations_norm)} normalization violations (_->/)!')
    for o, n in violations_norm[:5]:
        print(f'  {o} -> {n}')
else:
    print('\nOK: normalize_lipid_name preserves _ in all names.')

goslin_Morita.csv loaded for formula-level fallback.
Total unique Morita lipid names: 1289

Normalization examples:
  Cer d18:1_14:0                      -> Cer 18:1;2_14:0
  Cer d18:1_16:0                      -> Cer 18:1;2_16:0
  Cer d18:1_16:1                      -> Cer 18:1;2_16:1
  Cer d18:1_18:0                      -> Cer 18:1;2_18:0
  Cer d18:1_18:1                      -> Cer 18:1;2_18:1
  Cer d18:1_18:2                      -> Cer 18:1;2_18:2

OK: normalize_lipid_name preserves _ in all names.


## Step 3 — Cascade matching (SwissLipids-adapted strategies)

SwissLipids uses different abbreviation formats from LipidMaps.
Strategies are tried in priority order.

**`_` vs `/` separator rule:**
- For most lipid classes, `_`-input generates `_`-keyed lookups (and vice versa).
  SwissLipids contains both `PC(16:0_18:1)` and `PC(16:0/18:1)` as distinct entries.
- **Exception — sphingolipids (Cer, SM, HexCer, ...):**
  SwissLipids registers these with `/` only (verified: 0 entries with `_`, 2000+ with `/`).
  Biologically, the sphingoid base sn-position is fixed by the amide bond, so
  SwissLipids treats all ceramide molecular subspecies as structural subspecies.
  `to_cer_d_slash` therefore converts `_` input to `/` DB key intentionally.

| Priority | Strategy | Input → SwissLipids DB key | Sep rule |
|---|---|---|---|
| 1 | Exact | `PC 16:0_18:1` → `PC 16:0_18:1` | preserves `_` |
| 2 | Paren + underscore | `PC 16:0_18:1` → `PC(16:0_18:1)` | `_`-input only |
| 3 | Paren + slash | `PC 16:0/18:1` → `PC(16:0/18:1)` | `/`-input only |
| 4 | Cer d-notation + slash | `Cer 18:1;2_16:0` → `Cer(d18:1/16:0)` | `_`→`/` (sphingolipid exception) |
| 5 | Lyso with `0:0` | `LPC 16:0` → `LPC(16:0_0:0)` | single-chain |

Since Morita data uses `_` exclusively, strategy 3 (`paren_slash`) is effectively inactive for this dataset.


In [43]:
SPHINGOLIPID_CLASSES = {'Cer', 'SM', 'HexCer', 'Hex2Cer', 'LacCer', 'GlcCer'}
LYSO_CLASSES         = {'LPC', 'LPE', 'LPG', 'LPI', 'LPS', 'LPA'}


def _split_chains(name):
    """Return (class_prefix, [chain_strings], sep) for a normalized lipid name.

    sep = '/' — sn-position specified (Structural subspecies, LIPID MAPS '/')
    sep = '_' — chain composition only (Molecular subspecies, LIPID MAPS '_')
    sep = None — single-chain lipid (no separator present)

    Preserving sep is critical: converting '/' to '_' or vice versa changes the
    structural level and causes incorrect database matches.
    """
    m = re.match(r'^([A-Za-z]+(?:\s[A-Za-z]+)?)\s+(.+)$', name)
    if not m:
        return None, [], None
    cls       = m.group(1)
    chain_str = m.group(2)

    if '/' in chain_str:
        sep    = '/'
        chains = [c.strip() for c in chain_str.split('/')]
    elif re.search(r'\d_\d', chain_str):
        sep    = '_'
        chains = [c.strip() for c in re.split(r'(?<=\d)_(?=\d)', chain_str)]
    else:
        sep    = None
        chains = [chain_str.strip()]

    return cls, chains, sep


def to_parentheses_underscore(name):
    """Molecular subspecies: 'PC 16:0_18:1' -> 'PC(16:0_18:1)' (sn-position unknown)."""
    cls, chains, sep = _split_chains(name)
    if cls is None or len(chains) < 2 or sep != '_':
        return None
    return f"{cls}({'_'.join(chains)})"


def to_parentheses_slash(name):
    """Structural subspecies: 'PC 16:0/18:1' -> 'PC(16:0/18:1)' (sn-position known)."""
    cls, chains, sep = _split_chains(name)
    if cls is None or len(chains) < 2 or sep != '/':
        return None
    return f"{cls}({'/'.join(chains)})"


def to_cer_d_slash(name):
    """
    Convert sphingolipid ';2' notation to SwissLipids 'd'-prefix with slash separator.
    'Cer 18:1;2_16:0' -> 'Cer(d18:1/16:0)'
    'SM  18:1;2_18:0' -> 'SM(d18:1/18:0)'

    WHY '/' IS USED DESPITE '_' INPUT (intentional exception):
    SwissLipids registers ALL sphingolipid molecular subspecies with '/' in d-notation
    (verified: 0 entries with '_', 2000+ entries with '/' for Cer/SM alone).
    Biologically, the sphingoid base sn-position in ceramides is fixed by the amide
    bond to the fatty acid — SwissLipids therefore treats these as structural subspecies
    and stores them with '/'.  Using '_' here would yield zero matches.

    This is the ONLY place in the cascade where '_' input generates a '/'-keyed lookup.
    All glycerophospholipids (PC, PE, TG, etc.) have both '_' and '/' entries in
    SwissLipids, so they are correctly matched via to_parentheses_underscore.
    """
    cls, chains, sep = _split_chains(name)
    if cls is None or len(chains) < 2 or cls not in SPHINGOLIPID_CLASSES:
        return None
    base = re.sub(r';.*', '', chains[0])  # strip ';2' or ';O2'
    return f"{cls}(d{base}/{'/'.join(chains[1:])})"


def to_lyso_underscore(name):
    """
    Lysophospholipid -> SwissLipids format with explicit '0:0' for missing chain.
    'LPC 16:0' -> 'LPC(16:0_0:0)'
    Why: SwissLipids represents lyso lipids with an explicit zero chain.
    """
    cls, chains, sep = _split_chains(name)
    if cls is None or cls not in LYSO_CLASSES:
        return None
    if len(chains) == 1:
        return f"{cls}({chains[0]}_0:0)"
    return None


def cascade_lookup(name):
    """
    Try each name-format conversion in priority order.
    Return the first hit found in lipid_db, or None if nothing matches.

    Note: for most lipid classes, '_' input stays '_' in the DB key.
    Exception: sphingolipids (Cer, SM, HexCer, ...) always use '/' in SwissLipids
    d-notation — see to_cer_d_slash for the biological justification.
    """
    strategies = [
        ('exact',       lambda n: n),
        ('paren_under', to_parentheses_underscore),
        ('paren_slash', to_parentheses_slash),
        ('cer_d_slash', to_cer_d_slash),
        ('lyso',        to_lyso_underscore),
    ]
    for label, fn in strategies:
        converted = fn(name)
        if converted and converted in lipid_db:
            entries = lipid_db[converted]
            seen_levels = {}
            for e in entries:
                lv = e.get('level', '')
                if lv:
                    seen_levels[lv] = None
            return {
                'sl_ids':         [e['sl_id']  for e in entries],
                'sl_smiles_list': [e['smiles'] for e in entries],
                'converted_name': converted,
                'strategy':       label,
                'levels':         list(seen_levels.keys()),
            }
    return None


match_results = [cascade_lookup(n) for n in lipid_names_norm]


## Step 4 — Match rate and breakdown

In [44]:
n_total   = len(lipid_names)
n_matched = sum(1 for r in match_results if r is not None)
n_failed  = n_total - n_matched

strategy_counts = Counter(
    r['strategy'] for r in match_results if r is not None
)

print(f'Total lipid names  : {n_total}')
print(f'Matched            : {n_matched}  ({n_matched / n_total * 100:.1f}%)')
print(f'Unmatched          : {n_failed}   ({n_failed  / n_total * 100:.1f}%)')
print()
print('Match breakdown by strategy:')
for strat, cnt in strategy_counts.most_common():
    print(f'  {strat:<18} {cnt:>5}  ({cnt / n_total * 100:.1f}%)')

unmatched = [name for name, r in zip(lipid_names, match_results) if r is None]
print(f'\nUnmatched names ({len(unmatched)}):')
for name in unmatched:
    print(f'  {name}')

Total lipid names  : 1289
Matched            : 1213  (94.1%)
Unmatched          : 76   (5.9%)

Match breakdown by strategy:
  paren_under         1136  (88.1%)
  lyso                  43  (3.3%)
  cer_d_slash           34  (2.6%)

Unmatched names (76):
  MG 14:0
  MG 16:0
  MG 16:1
  MG 18:0
  MG 18:1
  MG 18:2
  MG 20:0
  MG 20:1
  MG 20:2
  MG 20:3
  MG 20:5
  MG 22:0
  MG 22:6
  SM 30:1
  SM 32:1
  SM 32:2
  SM 34:1
  SM 34:2
  SM 36:1
  SM 36:2
  SM 36:3
  SM 36:4
  SM 36:5
  SM 38:1
  SM 38:2
  SM 38:3
  SM 38:4
  SM 38:5
  SM 38:6
  SM 40:1
  SM 40:2
  SM 40:3
  SM 40:4
  SM 40:5
  SM 40:6
  SM 40:7
  SM 42:1
  SM 42:2
  SM 44:1
  CE(16:1)
  CE(18:0)
  CE(18:1)
  CE(18:2)
  CE(18:3)
  CE(20:0)
  CE(20:1)
  CE(20:2)
  CE(20:3)
  CE(20:4)
  CE(20:5)
  CE(22:3)
  CE(22:4)
  CE(22:5)
  CE(22:6)
  FA(12:0)
  FA(14:0)
  FA(14:1)
  FA(16:0)
  FA(16:1)
  FA(18:0)
  FA(18:1)
  FA(18:2)
  FA(18:3)
  FA(18:4)
  FA(20:1)
  FA(20:2)
  FA(20:3)
  FA(20:4)
  FA(20:5)
  FA(22:1)
  FA(22:4)
  FA(

## Step 5 — Compare with LipidMaps match results

Load `lipidmaps_Morita.csv` to identify which lipids are newly matched
by SwissLipids that were absent from LipidMaps.

In [45]:
# Build a name -> SwissLipids result dict
sl_result_map = {
    name: result
    for name, result in zip(lipid_names, match_results)
}

try:
    df_lm = pd.read_csv('../data/processed/lipidmaps_Morita.csv')[
        ['original_name', 'converted_name', 'strategy']
    ].rename(columns={'converted_name': 'lm_converted', 'strategy': 'lm_strategy'})

    df_lm['sl_matched'] = df_lm['original_name'].map(
        lambda n: sl_result_map.get(n) is not None
    )
    df_lm['lm_matched'] = df_lm['lm_converted'].notna()

    only_sl = df_lm[df_lm['sl_matched'] & ~df_lm['lm_matched']]
    only_lm = df_lm[~df_lm['sl_matched'] & df_lm['lm_matched']]
    both    = df_lm[ df_lm['sl_matched'] &  df_lm['lm_matched']]
    neither = df_lm[~df_lm['sl_matched'] & ~df_lm['lm_matched']]

    print('Coverage comparison:')
    print(f'  Both LipidMaps AND SwissLipids : {len(both)}')
    print(f'  SwissLipids only               : {len(only_sl)}')
    print(f'  LipidMaps only                 : {len(only_lm)}')
    print(f'  Neither                        : {len(neither)}')
    print()

    if len(only_sl) > 0:
        print(f'Lipids uniquely matched by SwissLipids ({len(only_sl)}):')
        print(only_sl[['original_name']].to_string(index=False))
        print()

    if len(neither) > 0:
        print(f'Lipids matched by neither LipidMaps nor SwissLipids ({len(neither)}):')
        print(neither[['original_name']].to_string(index=False))

except FileNotFoundError:
    print('lipidmaps_Morita.csv not found — skipping comparison.')

Coverage comparison:
  Both LipidMaps AND SwissLipids : 1197
  SwissLipids only               : 16
  LipidMaps only                 : 62
  Neither                        : 14

Lipids uniquely matched by SwissLipids (16):
    original_name
   Cer d18:1_18:3
   Cer d18:1_18:4
   Cer d18:1_20:3
   Cer d18:1_20:4
   Cer d18:1_20:5
   Cer d18:1_22:2
   Cer d18:1_22:3
   Cer d18:1_22:4
   Cer d18:1_22:5
   Cer d18:1_22:6
         LPC 22:3
     PE 22:5_22:6
     PG 22:4_22:5
     PG 22:5_22:6
HexCer d18:1_22:2
HexCer d18:1_22:3

Lipids matched by neither LipidMaps nor SwissLipids (14):
original_name
      MG 14:0
      MG 16:1
      MG 20:1
      MG 20:2
      MG 22:0
      SM 36:4
      SM 36:5
      SM 38:4
      SM 38:5
      SM 38:6
      SM 40:4
      SM 40:5
      SM 40:6
      SM 40:7


## Step 6 — Save results

In [46]:
# Manual level for lipids unmatched by both LipidMaps and SwissLipids
# but whose hierarchy level is unambiguous from the lipid name itself.
MANUAL_LEVEL = {
    'SM 40:4': 'Species',
    'SM 40:5': 'Species',
    'SM 40:6': 'Species',
    'SM 40:7': 'Species',
}

rows = []
for orig, norm, result in zip(lipid_names, lipid_names_norm, match_results):
    if result:
        rows.append({
            'original_name':   orig,
            'normalized_name': norm,
            'converted_name':  result['converted_name'],
            'strategy':        result['strategy'],
            'sl_ids':          ';'.join(result['sl_ids']),
            'n_sl_ids':        len(result['sl_ids']),
            'levels':          ';'.join(result['levels']),
        })
    else:
        rows.append({
            'original_name':   orig,
            'normalized_name': norm,
            'converted_name':  None,
            'strategy':        None,
            'sl_ids':          None,
            'n_sl_ids':        0,
            'levels':          MANUAL_LEVEL.get(orig),
        })

df_result = pd.DataFrame(rows)

out_path = '../data/processed/swisslipids_Morita.csv'
df_result.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {df_result.shape}')
df_result.head(10)

Saved: ../data/processed/swisslipids_Morita.csv
Shape: (1289, 7)


,original_name,normalized_name,converted_name,strategy,sl_ids,n_sl_ids,levels
0,Cer d18:1_14:0,Cer 18:1;2_14:0,Cer(d18:1/14:0),cer_d_slash,SLM:000392128,1,Molecular subspecies
1,Cer d18:1_16:0,Cer 18:1;2_16:0,Cer(d18:1/16:0),cer_d_slash,SLM:000392131,1,Molecular subspecies
2,Cer d18:1_16:1,Cer 18:1;2_16:1,Cer(d18:1/16:1),cer_d_slash,SLM:000392132,1,Molecular subspecies
3,Cer d18:1_18:0,Cer 18:1;2_18:0,Cer(d18:1/18:0),cer_d_slash,SLM:000392135,1,Molecular subspecies
4,Cer d18:1_18:1,Cer 18:1;2_18:1,Cer(d18:1/18:1),cer_d_slash,SLM:000392136,1,Molecular subspecies
5,Cer d18:1_18:2,Cer 18:1;2_18:2,Cer(d18:1/18:2),cer_d_slash,SLM:000392137,1,Molecular subspecies
6,Cer d18:1_18:3,Cer 18:1;2_18:3,Cer(d18:1/18:3),cer_d_slash,SLM:000392138,1,Molecular subspecies
7,Cer d18:1_18:4,Cer 18:1;2_18:4,Cer(d18:1/18:4),cer_d_slash,SLM:000392139,1,Molecular subspecies
8,Cer d18:1_20:0,Cer 18:1;2_20:0,Cer(d18:1/20:0),cer_d_slash,SLM:000392142,1,Molecular subspecies
9,Cer d18:1_20:1,Cer 18:1;2_20:1,Cer(d18:1/20:1),cer_d_slash,SLM:000392143,1,Molecular subspecies


## Step 7 — SwissLipids hierarchy level breakdown

Count how many Morita et al. lipids fall into each SwissLipids hierarchy level.

SwissLipids levels (finest to coarsest):
- `Isomeric subspecies` — full stereo info
- `Structural subspecies` — sn-position known
- `Molecular subspecies` — chain composition known, sn unknown
- `Species` — only class + total C:DB known

A single matched name may map to multiple levels; the finest (most specific) level is used.  
`(unmatched)` = not found in SwissLipids (and no manual level assigned).

In [47]:
# SwissLipids levels (finest to coarsest)
LEVEL_ORDER = [
    'Isomeric subspecies',
    'Structural subspecies',
    'Molecular subspecies',
    'Species',
    'Class',
    'Category',
]
LEVEL_RANK = {lv: i for i, lv in enumerate(LEVEL_ORDER)}


def finest_level(levels_str):
    """Return the finest (most specific) level from a semicolon-separated string."""
    if pd.isna(levels_str):
        return None
    levels = [lv.strip() for lv in levels_str.split(';') if lv.strip()]
    ranked = sorted(levels, key=lambda lv: LEVEL_RANK.get(lv, 99))
    return ranked[0] if ranked else None


df_result['finest_level'] = df_result['levels'].apply(finest_level)

print('finest_level distribution:')
print(df_result['finest_level'].value_counts(dropna=False).to_string())

finest_level distribution:
finest_level
Molecular subspecies    1213
NaN                       72
Species                    4


In [48]:
# Overall level summary table
n_total = len(df_result)
level_counts = df_result['finest_level'].value_counts(dropna=False)

rows_summary = []
for lv in LEVEL_ORDER:
    n = int(level_counts.get(lv, 0))
    rows_summary.append({'level': lv, 'n': n, 'pct': f'{n / n_total * 100:.1f}%'})

n_unmatched = int(df_result['finest_level'].isna().sum())
rows_summary.append({'level': '(unmatched)', 'n': n_unmatched, 'pct': f'{n_unmatched / n_total * 100:.1f}%'})

df_level_summary = pd.DataFrame(rows_summary)
print(f'SwissLipids level summary (n={n_total}):')
print()
print(df_level_summary.to_string(index=False))

SwissLipids level summary (n=1289):

                level    n   pct
  Isomeric subspecies    0  0.0%
Structural subspecies    0  0.0%
 Molecular subspecies 1213 94.1%
              Species    4  0.3%
                Class    0  0.0%
             Category    0  0.0%
          (unmatched)   72  5.6%


In [49]:
# Breakdown by lipid class (uses df_goslin loaded in Step 2)
df2 = df_result.merge(df_goslin[['original_name', 'class_name']], on='original_name', how='left')

pivot = (
    df2.groupby(['class_name', 'finest_level'], dropna=False)
    .size()
    .unstack(fill_value=0)
)

col_order = [lv for lv in LEVEL_ORDER if lv in pivot.columns]
if None in pivot.columns:
    col_order.append(None)
pivot = pivot.reindex(columns=col_order, fill_value=0)
pivot.columns = [str(c) if c is not None else '(unmatched)' for c in pivot.columns]

pivot['total'] = pivot.sum(axis=1)
pivot = pivot.sort_values('total', ascending=False)

print('Lipid count per class x SwissLipids level:')
pivot

Lipid count per class x SwissLipids level:


,Molecular subspecies,Species,nan,total
class_name,,,,
TG,638,0,0,638
DG,138,0,0,138
PE,93,0,0,93
PC,88,0,0,88
PG,61,0,0,61
PI,56,0,0,56
PS,38,0,0,38
SM,0,4,22,26
Cer,24,0,0,24
